# The goal of this notebook is to test the component-drift mechanism. 
Preferably, the notebook explores the following: 
- How does the model react to drifts in the signal?
- How does the model reach to sudden changes in noise?
- How does the model compare to a version without sEM?
- Hoew does the model-parameters evole over time?

### Test design
**Data**\ 
use a single image from the dataset. NAME=HYPSO-1_HSI_20221021T093938Z-l1a\
Trainingdata = first ten rows (10*684=6,840 pixels)
Testdata = remaining rows ((956-10)*684=647,064 pixels), at every 30 row, the light intensity is increased





In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as patches
import glob
import torch
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from otfp import MFA_OTFP
from hypso import Hypso


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

# We now only need the primary Svalbard image and the Forest image
file_path = "../data/uddevold_2022-10-21T09-39-38Z-l1a.nc"

satobj = Hypso(file_path)
cube = getattr(satobj, "l1b_cube", None)
if cube is not None:
    image = cube.values.astype(np.float32)
    print(f"Successfully loaded {file_path.split('/')[-1]} | Shape: {image.shape}")

h,w,b = image.shape
print(f"Image Dimensions: Height={h}, Width={w}, Bands={b}")
    
    
    #mid_row = svalbard_full.shape[0] // 2
    #
    #img_train = svalbard_full[:mid_row, :, :]
    ## Using mid_row:mid_row*2 ensures identical shapes even if the original row count was odd
    #img_test_base = svalbard_full[mid_row:mid_row*2, :, :] 
    #
    #print(f"Splitting primary image in half:")
    #print(f"New Train Shape (Top Half):     {img_train.shape}")
    #print(f"New Test Shape (Bottom Half):   {img_test_base.shape}")
    #print(f"Forest Source Shape:            {img_forest.shape}")
#
    ## --- Helper Function: Hyperspectral to RGB ---
    #def get_rgb(hsi_cube):
    #    """Extracts RGB, normalizes globally to preserve color balance, and applies gamma correction."""
    #    r, g, b = hsi_cube[:, :, 70], hsi_cube[:, :, 50], hsi_cube[:, :, 20]
    #    rgb = np.stack([r, g, b], axis=-1)
    #    
    #    p_low = np.nanpercentile(rgb, 2)
    #    p_high = np.nanpercentile(rgb, 98)
    #    rgb_norm = (rgb - p_low) / (p_high - p_low + 1e-8)
    #    rgb_clipped = np.clip(rgb_norm, 0, 1)
    #    
    #    gamma = 0.6  
    #    rgb_gamma = rgb_clipped ** gamma
    #    
    #    return rgb_gamma
#
    ## --- 2. Setup Patch Coordinates ---
    #PATCH_SIZE = 100
    #
    ## Where to sample FROM in the forest image
    #f_row, f_col = 230, 870 
    #
    ## Dynamically center the insertion box inside the testing half
    #t_row = (img_test_base.shape[0] - PATCH_SIZE) // 2
    #t_col = (img_test_base.shape[1] - PATCH_SIZE) // 2
#
    ## --- 3. Extract the Forest Sample ---
    #forest_patch = img_forest[f_row:f_row+PATCH_SIZE, f_col:f_col+PATCH_SIZE, :].copy()
#
    ## --- 4. Create the Two Test Scenarios ---
    #
    ## A) Novel Material Spawning (Forest injected into Test Image)
    #test_img_forest = img_test_base.copy()
    #test_img_forest[t_row:t_row+PATCH_SIZE, t_col:t_col+PATCH_SIZE, :] = forest_patch
    #
    ## B) Noise Rejection (Gaussian White Noise injected into Test Image)
    #test_img_noise = img_test_base.copy()
    #noise_patch = test_img_noise[t_row:t_row+PATCH_SIZE, t_col:t_col+PATCH_SIZE, :].copy()
    #
    #np.random.seed(42) 
    ## Add severe Gaussian white noise across ALL bands to destroy the spectral angle
    #gaussian_noise = np.random.normal(loc=0.0, scale=np.nanstd(img_test_base) * 2, size=noise_patch.shape)
    #noise_patch = noise_patch + gaussian_noise
    #
    ## Clip it to keep it within realistic sensor boundaries
    #test_img_noise[t_row:t_row+PATCH_SIZE, t_col:t_col+PATCH_SIZE, :] = np.clip(noise_patch, 0, np.nanmax(img_test_base))
#
    ## --- 5. Plotting the Validation Grid ---
    #fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    #
    ## Plot 1: Pre-training Image
    #axes[0].imshow(get_rgb(img_train))
    #axes[0].set_title("1. Training Image\n(Top Half Baseline)", fontsize=14)
    #axes[0].axis('off')
    #
    ## Plot 2: Source Forest Image
    #axes[1].imshow(get_rgb(img_forest))
    #axes[1].set_title("2. Source Forest Image\n(Red Box = Extracted Sample)", fontsize=14)
    #rect_source = patches.Rectangle((f_col, f_row), PATCH_SIZE, PATCH_SIZE, 
    #                                linewidth=2.5, edgecolor='red', facecolor='none')
    #axes[1].add_patch(rect_source)
    #axes[1].axis('off')
    #
    ## Plot 3: Noise Rejection Test
    #axes[2].imshow(get_rgb(test_img_noise))
    #axes[2].set_title("3. Test Stream: Noise Rejection\n(Bottom Half - Expected: 0 Spawns)", fontsize=14)
    #rect_noise = patches.Rectangle((t_col, t_row), PATCH_SIZE, PATCH_SIZE, 
    #                               linewidth=2.5, edgecolor='yellow', facecolor='none')
    #axes[2].add_patch(rect_noise)
    #axes[2].axis('off')
    #
    ## Plot 4: Novel Material Test
    #axes[3].imshow(get_rgb(test_img_forest))
    #axes[3].set_title("4. Test Stream: Novel Material\n(Bottom Half - Expected: 1 Spawn)", fontsize=14)
    #rect_target = patches.Rectangle((t_col, t_row), PATCH_SIZE, PATCH_SIZE, 
    #                                linewidth=2.5, edgecolor='lime', facecolor='none')
    #axes[3].add_patch(rect_target)
    #axes[3].axis('off')
    #
    #plt.tight_layout()
    #plt.show()
else:
    print("Data loading failed. Check your file paths and Hypso class configuration.")